[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/38_causal_machine_learning/04_meta_learners_and_uplift.ipynb)

# Bài 04 — Meta-learner và uplift modeling

**🎯 Mục tiêu**
- **Hiệu ứng dị biệt** (CATE) $\tau(x)$: coupon tác động khác nhau với từng người — ai nên nhận?
- **Meta-learner**: S-learner, T-learner, X-learner — dùng bất kỳ mô hình ML nào làm "khối xây dựng"; đánh giá bằng **PEHE** khi biết hiệu ứng thật.
- **Uplift modeling** trên dữ liệu thí nghiệm: đường **Qini**, chính sách nhắm mục tiêu theo **lợi nhuận**.
- Bài học quan trọng nhất: **dự đoán phản hồi ≠ dự đoán tác động**.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/38_causal_machine_learning

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from ml_course.datasets import make_causal_marketing
from ml_course.visualization import set_style

set_style()
warnings.filterwarnings("ignore")
FEATURES = ["age", "income_mvnd", "past_purchases"]
gbm = lambda: HistGradientBoostingRegressor(random_state=0)

## 1. Hiệu ứng dị biệt

ATE trả lời "trung bình coupon có hiệu quả không?". Nhưng phát coupon tốn tiền — câu hỏi kinh doanh là **phát cho ai**: những người có $\tau(x) = \mathbb E[Y(1) - Y(0)\mid X = x]$ đủ lớn. Trong dữ liệu mô phỏng, hiệu ứng thật giảm theo tuổi (người trẻ phản ứng mạnh với coupon).

In [ ]:
obs = make_causal_marketing(20000, seed=1)                             # dữ liệu quan sát (có confounding)
idx_train, idx_test = train_test_split(np.arange(len(obs)), test_size=0.3, random_state=0)
X, T, Y, tau = obs[FEATURES].values, obs.coupon.values, obs.spend_kvnd.values, obs.true_effect.values
Xtr, Ttr, Ytr, Xte, tau_te = X[idx_train], T[idx_train], Y[idx_train], X[idx_test], tau[idx_test]
plt.figure(figsize=(6, 3.2))
plt.scatter(obs.age[:3000], obs.true_effect[:3000], s=3)
plt.xlabel("tuổi")
plt.ylabel("hiệu ứng thật của coupon τ (nghìn đồng)")
plt.tight_layout()
plt.show()

## 2. Ba meta-learner

- **S-learner** ("single"): một mô hình $\hat\mu(x, t)$ với $T$ là một đặc trưng; $\hat\tau(x) = \hat\mu(x, 1) - \hat\mu(x, 0)$. Đơn giản; nhưng mô hình có thể "bỏ qua" $T$ nếu hiệu ứng nhỏ so với nhiễu (regularization kéo hiệu ứng về 0).
- **T-learner** ("two"): hai mô hình riêng $\hat\mu_1$ (trên nhóm điều trị), $\hat\mu_0$ (nhóm đối chứng); $\hat\tau = \hat\mu_1 - \hat\mu_0$. Linh hoạt, nhưng hiệu của hai mô hình độc lập rất nhiễu — nhất là khi một nhóm nhỏ.
- **X-learner** (Künzel et al., 2019): bước 1 như T-learner; bước 2 **gán hiệu ứng** cho từng người bằng phản thực tế dự đoán ($D^1_i = Y_i - \hat\mu_0(x_i)$ với người được điều trị, $D^0_i = \hat\mu_1(x_i) - Y_i$ với người không) và fit mô hình cho $D$; bước 3 trộn theo điểm xu hướng $\hat\tau = e(x)\hat\tau_0(x) + (1 - e(x))\hat\tau_1(x)$. Mạnh khi hai nhóm lệch cỡ.

Với dữ liệu quan sát, mọi meta-learner vẫn cần giả định không có confounder chưa đo (các confounder phải nằm trong $X$).

In [ ]:
def s_learner(Xtr, Ttr, Ytr, Xte):
    m = gbm().fit(np.column_stack([Xtr, Ttr]), Ytr)
    return m.predict(np.column_stack([Xte, np.ones(len(Xte))])) - m.predict(np.column_stack([Xte, np.zeros(len(Xte))]))


def t_learner(Xtr, Ttr, Ytr, Xte):
    m1 = gbm().fit(Xtr[Ttr == 1], Ytr[Ttr == 1])
    m0 = gbm().fit(Xtr[Ttr == 0], Ytr[Ttr == 0])
    return m1.predict(Xte) - m0.predict(Xte)


def x_learner(Xtr, Ttr, Ytr, Xte):
    m1 = gbm().fit(Xtr[Ttr == 1], Ytr[Ttr == 1])
    m0 = gbm().fit(Xtr[Ttr == 0], Ytr[Ttr == 0])
    d1 = Ytr[Ttr == 1] - m0.predict(Xtr[Ttr == 1])                     # hiệu ứng gán cho người được điều trị
    d0 = m1.predict(Xtr[Ttr == 0]) - Ytr[Ttr == 0]                     # ... và cho người không được điều trị
    tau1 = gbm().fit(Xtr[Ttr == 1], d1).predict(Xte)
    tau0 = gbm().fit(Xtr[Ttr == 0], d0).predict(Xte)
    e = LogisticRegression(max_iter=1000).fit(Xtr, Ttr).predict_proba(Xte)[:, 1]
    return e * tau0 + (1 - e) * tau1


rows = []
cate = {}
for name, learner in [("S-learner", s_learner), ("T-learner", t_learner), ("X-learner", x_learner)]:
    cate[name] = learner(Xtr, Ttr, Ytr, Xte)
    rows.append({"meta-learner": name, "PEHE (√MSE với τ thật)": np.sqrt(np.mean((cate[name] - tau_te) ** 2)),
                 "tương quan với τ thật": np.corrcoef(cate[name], tau_te)[0, 1], "ATE ước lượng": cate[name].mean()})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print(f"ATE thật (tập test) = {tau_te.mean():.1f} | độ lệch chuẩn của τ thật = {tau_te.std():.1f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, (name, c) in zip(axes, cate.items()):
    ax.scatter(tau_te[:2000], c[:2000], s=3, alpha=0.4)
    ax.plot([tau_te.min(), tau_te.max()], [tau_te.min(), tau_te.max()], "k:")
    ax.set_title(name)
    ax.set_xlabel("τ thật")
axes[0].set_ylabel("τ ước lượng")
plt.tight_layout()
plt.show()

Cả ba nắm được xu hướng (tương quan cao với hiệu ứng thật); T-learner nhiễu nhất (hiệu của hai mô hình độc lập). Trên dữ liệu này — hiệu ứng đơn giản (tuyến tính theo tuổi), lớn so với nhiễu — S-learner làm tốt nhất; không có meta-learner nào thắng mọi bài toán (Künzel et al., 2019 cho ví dụ ngược lại). Ngoài đời không có $\tau$ thật để tính PEHE — phải đánh giá gián tiếp, như mục 3.

## 3. Uplift modeling trên dữ liệu thí nghiệm

Doanh nghiệp thường chạy **A/B test** (phát coupon ngẫu nhiên cho một phần khách hàng) để học mô hình uplift, rồi dùng nó chọn ai nhận coupon trong chiến dịch thật. Dữ liệu thí nghiệm không có confounding và cho phép **đánh giá** mô hình bằng **đường Qini**: sắp khách theo uplift dự đoán giảm dần; với top $k$ khách, ước lượng **số tiền tăng thêm** nếu chỉ phát cho họ:

$$Q(k) = \sum_{\text{top } k,\,T=1} Y - \sum_{\text{top } k,\,T=0} Y\cdot\frac{n_{1}(k)}{n_{0}(k)}$$

So sánh ba cách xếp hạng: **uplift dự đoán** (T-learner), **chi tiêu dự đoán** (mô hình phản hồi thông thường — "ai chi nhiều thì gửi"), và **ngẫu nhiên**; thêm **oracle** (xếp theo $\tau$ thật) làm trần.

In [ ]:
ab = make_causal_marketing(30000, seed=2, randomized=True)
a_train, a_test = train_test_split(np.arange(len(ab)), test_size=0.4, random_state=0)
XA, TA, YA, tauA = ab[FEATURES].values, ab.coupon.values, ab.spend_kvnd.values, ab.true_effect.values
uplift_pred = t_learner(XA[a_train], TA[a_train], YA[a_train], XA[a_test])
response_pred = gbm().fit(XA[a_train], YA[a_train]).predict(XA[a_test])     # "ai sẽ chi nhiều"


def qini_curve(score, y, t):
    order = np.argsort(-score)
    y, t = y[order], t[order]
    n1, n0 = np.cumsum(t), np.cumsum(1 - t)
    y1, y0 = np.cumsum(y * t), np.cumsum(y * (1 - t))
    return np.r_[0, y1 - y0 * np.divide(n1, n0, out=np.zeros(len(n0)), where=n0 > 0)]


y_test, t_test = YA[a_test], TA[a_test]
rankings = {"uplift dự đoán (T-learner)": uplift_pred, "chi tiêu dự đoán": response_pred,
            "ngẫu nhiên": np.random.default_rng(0).random(len(a_test)), "oracle (τ thật)": tauA[a_test]}
fig, ax = plt.subplots(figsize=(7, 4.2))
frac = np.linspace(0, 1, len(a_test) + 1)
for name, score in rankings.items():
    q = qini_curve(score, y_test, t_test)
    ax.plot(frac, q / 1000, label=name, ls=":" if "oracle" in name else "-")
    top20 = np.argsort(-score)[: int(0.2 * len(score))]
    print(f"{name:28s}: Qini tại top 20% = {q[int(0.2 * len(score))] / 1000:7.1f} triệu | hiệu ứng thật TB của top 20% = {tauA[a_test][top20].mean():6.1f}")
ax.set_xlabel("tỉ lệ khách hàng được nhắm (theo thứ tự xếp hạng)")
ax.set_ylabel("chi tiêu tăng thêm ước lượng (triệu)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

Xếp theo **uplift** gần sát oracle; xếp theo **chi tiêu dự đoán** chỉ hơn ngẫu nhiên một chút. Người chi tiêu nhiều (thu nhập cao, mua nhiều) **không** phải người phản ứng mạnh nhất với coupon — họ chi nhiều **dù có hay không** coupon. Mô hình phản hồi trả lời "ai sẽ mua", mô hình uplift trả lời "ai sẽ mua **thêm** vì ta tác động".

## 4. Chính sách nhắm mục tiêu theo lợi nhuận

Mỗi coupon tốn **100** nghìn đồng (chi phí + giảm giá). Chính sách tối ưu: phát cho người có $\tau(x) > 100$. Lợi nhuận thật (tính bằng $\tau$ thật, nghìn đồng/khách trên toàn tập test):

In [ ]:
COST = 100
tau_test = tauA[a_test]
n_target = int((uplift_pred > COST).sum())
policies = {
    "phát cho tất cả": np.ones(len(a_test), bool),
    "không phát cho ai": np.zeros(len(a_test), bool),
    "uplift dự đoán > chi phí": uplift_pred > COST,
    f"top {n_target:,} chi tiêu dự đoán (cùng số lượng)": response_pred >= np.sort(response_pred)[-n_target],
    "oracle: τ thật > chi phí": tau_test > COST,
}
rows = [{"chính sách": k, "tỉ lệ được phát": v.mean(), "lợi nhuận / khách (nghìn đồng)": np.sum(tau_test[v] - COST) / len(a_test)}
        for k, v in policies.items()]
print(pd.DataFrame(rows).round(2).to_string(index=False))

Chính sách theo uplift đạt gần mức oracle; phát cho tất cả lỗ ở những người có hiệu ứng nhỏ hơn chi phí; nhắm theo chi tiêu dự đoán — cách làm phổ biến trong thực tế — còn **tệ hơn phát cho tất cả**. Một khách "có giá trị cao" không phải khách "đáng tác động".

Lưu ý thực hành:
- Uplift học tốt nhất từ dữ liệu **thí nghiệm**; với dữ liệu quan sát cần các kỹ thuật bài 03 và giả định mạnh hơn.
- Có thể có người bị tác động **ngược** (sleeping dogs — gửi ưu đãi nhắc họ huỷ dịch vụ); mô hình uplift phát hiện được, mô hình phản hồi thì không.
- Kiểm chứng chính sách bằng một **A/B test mới** trước khi triển khai rộng.
- Thư viện: EconML (Microsoft: DML, causal forest, meta-learner), CausalML (Uber), scikit-uplift.

## 🧠 Kiểm tra nhanh

1. CATE khác ATE thế nào? Vì sao doanh nghiệp quan tâm CATE?
2. Mô tả S-, T-, X-learner. Ưu nhược của mỗi cách?
3. PEHE là gì? Vì sao ngoài đời không tính được?
4. Đường Qini được tính và đọc thế nào?
5. Vì sao nhắm mục tiêu theo "chi tiêu dự đoán cao" có thể tệ hơn phát ngẫu nhiên/phát cho tất cả?
6. Vì sao nên dùng dữ liệu thí nghiệm cho uplift modeling?

## 📌 Tóm tắt

- CATE $\tau(x)$ cho biết ai đáng tác động; meta-learner biến bất kỳ mô hình ML nào thành bộ ước lượng CATE (S/T/X), không cái nào luôn thắng.
- Uplift modeling + đường Qini trên dữ liệu A/B; chính sách theo uplift > chi phí tối đa lợi nhuận.
- Dự đoán phản hồi ≠ dự đoán tác động: khách giá trị cao chưa chắc là khách đáng tác động.

## 📚 Tài liệu tham khảo

- Künzel et al. (2019) — *Metalearners for estimating heterogeneous treatment effects using machine learning*
- Wager & Athey (2018) — *Estimation and Inference of Heterogeneous Treatment Effects using Random Forests* (causal forest)
- Radcliffe (2007) — *Using control groups to target on predicted lift: Building and assessing uplift models* (Qini)
- Gutierrez & Gérardy (2017) — *Causal Inference and Uplift Modelling: A Review of the Literature*
- [EconML](https://econml.azurewebsites.net/), [CausalML](https://causalml.readthedocs.io/)